# Skin Cancer Detection using HAM10000 Dataset
### MobileNetV2 Transfer Learning for 7-Class Skin Lesion Classification

**Classes:**
| Code | Full Name | 
|------|----------|
| akiec | Actinic Keratoses |
| bcc | Basal Cell Carcinoma |
| bkl | Benign Keratosis |
| df | Dermatofibroma |
| mel | Melanoma |
| nv | Melanocytic Nevi |
| vasc | Vascular Lesions |

## 1. Imports & Configuration

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from keras.applications import MobileNetV2
from keras.utils import to_categorical
from keras import layers, callbacks, optimizers, Sequential

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

In [ ]:
# ========================
# CONFIGURATION
# ========================
BASE_DIR = r'g:\Work\Cancer detection'
IMG_DIRS = [
    os.path.join(BASE_DIR, 'HAM10000_images_part_1'),
    os.path.join(BASE_DIR, 'HAM10000_images_part_2'),
]
METADATA_CSV = os.path.join(BASE_DIR, 'HAM10000_metadata.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = 224          # MobileNetV2 standard input
BATCH_SIZE = 32
PHASE1_EPOCHS = 10      # Feature extraction (frozen base)
PHASE2_EPOCHS = 15      # Fine-tuning (unfrozen top layers)
SEED = 42

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_FULL_NAMES = {
    'akiec': 'Actinic Keratoses',
    'bcc': 'Basal Cell Carcinoma',
    'bkl': 'Benign Keratosis',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic Nevi',
    'vasc': 'Vascular Lesions',
}

np.random.seed(SEED)
tf.random.set_seed(SEED)
import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## 2. GPU Check

In [ ]:
import keras
import torch
print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.config.backend()}")
if keras.config.backend() == "torch":
    if torch.cuda.is_available():
        print(f"SUCCESS: GPU detected: {torch.cuda.get_device_name(0)}")
        print("Training will run on the GPU.")
    else:
        print("WARNING: No GPU detected by PyTorch - training will run on CPU.")
        print("Please check your CUDA/PyTorch installation.")
else:
    print("WARNING: Keras is running on the TensorFlow backend.")
    print("NOTE: TensorFlow >= 2.11 does not support GPU on native Windows.")
    print("To run on the GPU, you must use the PyTorch backend.")
    print("Please RESTART your Jupyter kernel to apply the PyTorch backend.")


## 3. Data Loading & Exploration

In [ ]:
# Load metadata
df = pd.read_csv(METADATA_CSV)
print(f"Total records: {df.shape[0]}")
print(f"Unique lesions: {df['lesion_id'].nunique()}")
print(f"Unique images:  {df['image_id'].nunique()}")
print()

# Build image path lookup
img_path_map = {}
for img_dir in IMG_DIRS:
    for fname in os.listdir(img_dir):
        img_id = os.path.splitext(fname)[0]
        img_path_map[img_id] = os.path.join(img_dir, fname)

df['image_path'] = df['image_id'].map(img_path_map)
missing = df['image_path'].isnull().sum()
if missing > 0:
    print(f"WARNING: {missing} images not found on disk - dropping them")
    df = df.dropna(subset=['image_path'])

# Impute missing age with median
median_age = df['age'].median()
df['age'] = df['age'].fillna(median_age)
print(f"Missing ages imputed with median: {median_age}")

# Encode labels
le = LabelEncoder()
le.fit(CLASS_NAMES)  # Ensures consistent ordering
df['label'] = le.transform(df['dx'])

df.head(10)

In [ ]:
# Class distribution
print("Class distribution:")
for cls_name in CLASS_NAMES:
    count = (df['dx'] == cls_name).sum()
    pct = count / len(df) * 100
    print(f"  {cls_name:6s} ({CLASS_FULL_NAMES[cls_name]:25s}): {count:5d} ({pct:5.1f}%)")

### 3.1 Visualize Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
counts = df['dx'].value_counts()
colors = sns.color_palette("viridis", n_colors=7)
bars = ax.bar(
    [f"{c}\n({CLASS_FULL_NAMES[c]})" for c in counts.index],
    counts.values,
    color=colors,
    edgecolor='white',
    linewidth=0.5,
)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            str(count), ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_title('HAM10000 - Class Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Images', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150)
plt.show()

### 3.2 Sample Images per Class

In [ ]:
fig, axes = plt.subplots(2, 7, figsize=(21, 7))
for idx, cls_name in enumerate(CLASS_NAMES):
    subset = df[df['dx'] == cls_name].sample(n=2, random_state=SEED)
    for row_idx, (_, sample) in enumerate(subset.iterrows()):
        img = Image.open(sample['image_path']).resize((IMG_SIZE, IMG_SIZE))
        axes[row_idx, idx].imshow(img)
        axes[row_idx, idx].set_title(
            f"{cls_name.upper()}\n{CLASS_FULL_NAMES[cls_name]}",
            fontsize=8, fontweight='bold'
        )
        axes[row_idx, idx].axis('off')
plt.suptitle('Sample Images per Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Splitting (Lesion-Level to Prevent Leakage)

Some lesions have 2-6 images. If the same lesion appears in both train and test sets, the model gets an unfair advantage. We split at the **lesion level** to prevent this data leakage.

In [ ]:
# Get unique lesions with their diagnosis
lesion_df = df.groupby('lesion_id').agg({'dx': 'first', 'label': 'first'}).reset_index()

# Split lesions: 70% train, 15% val, 15% test
train_lesions, temp_lesions = train_test_split(
    lesion_df['lesion_id'],
    test_size=0.30,
    random_state=SEED,
    stratify=lesion_df['dx']
)

temp_dx = lesion_df[lesion_df['lesion_id'].isin(temp_lesions)]['dx']
val_lesions, test_lesions = train_test_split(
    temp_lesions,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_dx
)

# Map back to images
train_df = df[df['lesion_id'].isin(train_lesions)].reset_index(drop=True)
val_df = df[df['lesion_id'].isin(val_lesions)].reset_index(drop=True)
test_df = df[df['lesion_id'].isin(test_lesions)].reset_index(drop=True)

print(f"Train: {len(train_df):5d} images ({len(train_lesions)} lesions)")
print(f"Val:   {len(val_df):5d} images ({len(val_lesions)} lesions)")
print(f"Test:  {len(test_df):5d} images ({len(test_lesions)} lesions)")

## 5. Data Pipeline (tf.data with Augmentation)

In [ ]:
def load_and_preprocess_image(image_path, label):
    """Load an image from path, resize, and normalize."""
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label


def augment_image(image, label):
    """Apply data augmentation to training images."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    # Random rotation (0, 90, 180, or 270 degrees)
    k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k=k)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label


def create_dataset(dataframe, is_training=False):
    """Create a tf.data.Dataset from a dataframe."""
    paths = dataframe['image_path'].values
    labels = to_categorical(dataframe['label'].values, num_classes=7)

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

    if is_training:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
        dataset = dataset.shuffle(buffer_size=len(dataframe), seed=SEED)

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
print("Creating tf.data pipelines...")
train_ds = create_dataset(train_df, is_training=True)
val_ds = create_dataset(val_df, is_training=False)
test_ds = create_dataset(test_df, is_training=False)
print(f"Pipelines ready (batch size: {BATCH_SIZE})")

# Preview a batch
for images, labels in train_ds.take(1):
    print(f"Image batch shape: {images.shape}")
    print(f"Label batch shape: {labels.shape}")

## 6. Model Architecture - MobileNetV2 + Custom Head

In [ ]:
# Load pretrained MobileNetV2 (without top classification layer)
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
# Freeze all base layers for Phase 1
base_model.trainable = False

# Build classification head
model = Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(7, activation='softmax'),
])

print(f"Base model: MobileNetV2 ({len(base_model.layers)} layers, frozen)")
print(f"Total parameters:     {model.count_params():,}")
trainable_count = sum(np.prod(w.shape) for w in model.trainable_weights)
print(f"Trainable parameters: {trainable_count:,}")

In [ ]:
model.summary()

## 7. Class Weights (Handle Imbalance)

In [ ]:
train_labels = train_df['label'].values
weights = compute_class_weight('balanced', classes=np.arange(7), y=train_labels)
class_weights = dict(enumerate(weights))

print("Class weights (to handle imbalance):")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:6s}: {class_weights[i]:.3f}")

## 8. Phase 1 - Feature Extraction (Base Frozen)

Train **only the classification head** with the MobileNetV2 base completely frozen. This lets the new head learn to use the pretrained features.

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

phase1_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1,
    ),
    callbacks.ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, 'best_model_phase1.keras'),
        monitor='val_loss', save_best_only=True, verbose=1,
    ),
]

print("=" * 50)
print("PHASE 1: Feature Extraction (base frozen)")
print("=" * 50)

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    class_weight=class_weights,
    callbacks=phase1_callbacks,
    verbose=1,
)

## 9. Phase 2 - Fine-Tuning (Top Layers Unfrozen)

Unfreeze the **top 30 layers** of MobileNetV2 and train with a **very low learning rate** (1e-5) to fine-tune the features for skin lesion classification without destroying pretrained weights.

In [ ]:
# Unfreeze top 30 layers of MobileNetV2
base_model.trainable = True
fine_tune_from = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_from]:
    layer.trainable = False

trainable_count = sum(np.prod(w.shape) for w in model.trainable_weights)
print(f"Trainable parameters after unfreezing: {trainable_count:,}")

# Recompile with very low learning rate
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

phase2_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1,
    ),
    callbacks.ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, 'best_model_phase2.keras'),
        monitor='val_loss', save_best_only=True, verbose=1,
    ),
]

print("=" * 50)
print("PHASE 2: Fine-Tuning (top 30 layers unfrozen)")
print("=" * 50)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    class_weight=class_weights,
    callbacks=phase2_callbacks,
    verbose=1,
)

## 10. Training Curves

In [ ]:
# Combine histories from both phases
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']
epochs_range = range(1, len(acc) + 1)
phase1_end = len(history1.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy
ax1.plot(epochs_range, acc, 'b-', label='Train Accuracy', linewidth=2)
ax1.plot(epochs_range, val_acc, 'r-', label='Val Accuracy', linewidth=2)
ax1.axvline(x=phase1_end, color='gray', linestyle='--', alpha=0.7, label='Fine-tuning starts')
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(epochs_range, loss, 'b-', label='Train Loss', linewidth=2)
ax2.plot(epochs_range, val_loss, 'r-', label='Val Loss', linewidth=2)
ax2.axvline(x=phase1_end, color='gray', linestyle='--', alpha=0.7, label='Fine-tuning starts')
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
plt.show()

## 11. Evaluation on Test Set

In [ ]:
# Predict on test set
print("Predicting on test set...")
y_pred_proba = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_df['label'].values

### 11.1 Classification Report

In [ ]:
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3)
print("Classification Report")
print("=" * 60)
print(report)

# Save report
with open(os.path.join(OUTPUT_DIR, 'classification_report.txt'), 'w') as f:
    f.write("HAM10000 Skin Cancer Detection - Classification Report\n")
    f.write("=" * 60 + "\n\n")
    f.write(report)

### 11.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    ax=ax, linewidths=0.5, linecolor='white',
)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

### 11.3 ROC-AUC Curves (One-vs-Rest)

In [ ]:
y_true_onehot = to_categorical(y_true, num_classes=7)
fig, ax = plt.subplots(figsize=(10, 8))
colors = sns.color_palette("husl", 7)

for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_true_onehot[:, i], y_pred_proba[:, i])
    auc_score = roc_auc_score(y_true_onehot[:, i], y_pred_proba[:, i])
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{cls_name} (AUC = {auc_score:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
ax.set_title('ROC Curves (One-vs-Rest)', fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'roc_curves.png'), dpi=150)
plt.show()

In [ ]:
# Overall metrics
overall_auc = roc_auc_score(y_true_onehot, y_pred_proba, multi_class='ovr', average='weighted')
overall_acc = np.mean(y_pred == y_true)
print("=" * 40)
print(f"  Test Accuracy:     {overall_acc:.4f}")
print(f"  Weighted ROC-AUC:  {overall_auc:.4f}")
print("=" * 40)

### 11.4 Sample Predictions

In [ ]:
# Get one batch from test set
images, labels = next(iter(test_ds))
preds = model.predict(images, verbose=0)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for idx, ax in enumerate(axes.flat):
    if idx >= len(images):
        break
    ax.imshow(images[idx].numpy())
    true_label = CLASS_NAMES[np.argmax(labels[idx])]
    pred_label = CLASS_NAMES[np.argmax(preds[idx])]
    confidence = np.max(preds[idx]) * 100

    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(
        f"True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)",
        fontsize=10, fontweight='bold', color=color,
    )
    ax.axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_predictions.png'), dpi=150)
plt.show()

## 12. Export Model

In [ ]:
# Save the trained model
model_path = os.path.join(OUTPUT_DIR, 'skin_cancer_model.keras')
model.save(model_path)
print(f"Model saved: {model_path}")

# Save label mapping
label_mapping = {
    'class_names': CLASS_NAMES,
    'class_full_names': CLASS_FULL_NAMES,
    'label_to_class': {i: name for i, name in enumerate(CLASS_NAMES)},
    'class_to_label': {name: i for i, name in enumerate(CLASS_NAMES)},
    'metrics': {
        'accuracy': float(overall_acc),
        'weighted_auc': float(overall_auc),
    },
    'input_shape': [IMG_SIZE, IMG_SIZE, 3],
    'preprocessing': 'Resize to 224x224, normalize to [0,1]',
}
mapping_path = os.path.join(OUTPUT_DIR, 'label_mapping.json')
with open(mapping_path, 'w') as f:
    json.dump(label_mapping, f, indent=2)
print(f"Label mapping saved: {mapping_path}")

model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f"\nModel size: {model_size_mb:.1f} MB")

## 13. Quick Inference Demo

Load the saved model and predict on a single image.

In [ ]:
# Load saved model
import keras
loaded_model = keras.models.load_model(os.path.join(OUTPUT_DIR, 'skin_cancer_model.keras'))

# Pick a random test image
sample = test_df.sample(1, random_state=123).iloc[0]
img = Image.open(sample['image_path']).resize((IMG_SIZE, IMG_SIZE))
img_array = np.array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

# Predict
prediction = loaded_model.predict(img_array, verbose=0)
pred_class = CLASS_NAMES[np.argmax(prediction)]
pred_confidence = np.max(prediction) * 100
true_class = sample['dx']

# Display
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.imshow(img)
color = 'green' if pred_class == true_class else 'red'
ax1.set_title(f"True: {true_class} ({CLASS_FULL_NAMES[true_class]})\n"
              f"Predicted: {pred_class} ({pred_confidence:.1f}%)",
              fontsize=12, fontweight='bold', color=color)
ax1.axis('off')

# Prediction probabilities bar chart
probs = prediction[0]
bar_colors = ['green' if CLASS_NAMES[i] == pred_class else 'steelblue' for i in range(7)]
ax2.barh(CLASS_NAMES, probs, color=bar_colors)
ax2.set_xlabel('Probability')
ax2.set_title('Class Probabilities', fontsize=12, fontweight='bold')
ax2.set_xlim(0, 1)

plt.tight_layout()
plt.show()

print("\nAll class probabilities:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:6s} ({CLASS_FULL_NAMES[name]:25s}): {probs[i]*100:6.2f}%")

## Summary

### Output Files
All outputs are saved to the `output/` directory:
- `skin_cancer_model.keras` - Trained model
- `label_mapping.json` - Class labels and metadata
- `class_distribution.png` - Class distribution chart
- `sample_images.png` - Sample images per class
- `training_curves.png` - Accuracy/loss training curves
- `confusion_matrix.png` - Confusion matrix heatmap
- `roc_curves.png` - ROC-AUC curves
- `sample_predictions.png` - Sample prediction results
- `classification_report.txt` - Detailed metrics per class